# Baseline Model Conv2D (Shared Parameters)

## Import Libraries

In [1]:
import os

import pandas as pd
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))
tf.test.is_built_with_cuda()

I0000 00:00:1778304907.226744    6321 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


True

## Config

In [ ]:
TRAIN_DIR  = '../../data/intel/seg_train'
VAL_DIR    = '../../data/intel/seg_pred'
TEST_DIR   = '../../data/intel/seg_test'
MODEL_DIR  = '../../data/models/cnn'
os.makedirs(MODEL_DIR, exist_ok=True)

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 30
NUM_CLASSES = 6
LR          = 1e-3

## Import Dataset

In [ ]:
datagen = ImageDataGenerator(rescale=1.0/255.0)

train_gen = datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='sparse', shuffle=True, seed=SEED
)
val_gen = datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='sparse', shuffle=False
)
test_gen = datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='sparse', shuffle=False
)

print("Kelas :", train_gen.class_indices)
print(f"Train : {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}")

Found 14034 images belonging to 6 classes.
Found 3000 images belonging to 6 classes.


## CNN Model Baseline Architecture

In [3]:
model = keras.Sequential([
    keras.layers.Input(shape=((224, 224, 3))),
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='valid', name='conv2s_1'),
    keras.layers.MaxPooling2D((2, 2), name='maxpool_1'),
    keras.layers.Conv2D(64, (3, 3), activation='relu', padding='valid', name='conv2s_2'),
    keras.layers.MaxPooling2D((2, 2), name='maxpool_2'),
    keras.layers.Flatten(name='flatten'),
    keras.layers.Dense(128, activation='relu', name='dense_1'),
    keras.layers.Dense(6, activation='softmax', name='output'),
], name='Baseline_CNN')
model.summary()

I0000 00:00:1778304971.320482    6321 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "Baseline_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2s_1 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (MaxPooling2D)        │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2s_2 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_2 (MaxPooling2D)        │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,908,166 (91.20 MB)

 Trainable params: 23,908,166 (91.20 MB)

 Non-trainable params: 0 (0.00 B)

## Compilation and Training

In [4]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callback = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience = 5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='../../data/models/cnn/baseline_conv2d.h5',
        monitor='val_loss', save_best_only=True, verbose=1
    ),
]

history = model.fit(
    train_gen,
    epochs = EPOCHS,
    validation_data = val_gen,
    callbacks = callback,
    verbose = 1
)

Epoch 1/3


I0000 00:00:1778304975.065586    6321 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1778304977.412818    6553 service.cc:153] XLA service 0x72a214032460 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778304977.412886    6553 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.5.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1778304977.562665    6553 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778304978.077130    6553 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1778304978.128123    6553 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1550__.30
I0000 00:00:1778304992.050876    6553 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the proce

124/439 ━━━━━━━━━━━━━━━━━━━━ 2:13 425ms/step - accuracy: 0.3812 - loss: 1.6433

I0000 00:00:1778305045.275884    6554 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1550__.30


439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 440ms/step - accuracy: 0.5193 - loss: 1.2839

439/439 ━━━━━━━━━━━━━━━━━━━━ 257s 547ms/step - accuracy: 0.6197 - loss: 1.0293 - val_accuracy: 0.6773 - val_loss: 0.8404
Epoch 2/3
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 414ms/step - accuracy: 0.7422 - loss: 0.7244

439/439 ━━━━━━━━━━━━━━━━━━━━ 221s 502ms/step - accuracy: 0.7475 - loss: 0.7058 - val_accuracy: 0.7090 - val_loss: 0.7873
Epoch 3/3
439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step - accuracy: 0.8040 - loss: 0.5764

439/439 ━━━━━━━━━━━━━━━━━━━━ 201s 457ms/step - accuracy: 0.8036 - loss: 0.5688 - val_accuracy: 0.7573 - val_loss: 0.6618


## Evaluation

In [ ]:
y_pred_proba = model.predict(test_gen)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_gen.classes
macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f"Macro F1 Score: {macro_f1:.4f}")